# Pre-processing MultiplEYE Data

This notebook provides a step-by-step guide through how to process the eye-tracking data and the psychometric tests data collected within the MultiplEYE project. This goal of this notebook is twofold:

1. To provide a step-by-step guide on how to preprocess MultiplEYE data using the `pymovements` library and our custom preprocessing functions.
2. To serve as a tutorial for researchers who want to preprocess their own MultiplEYE data, or data from other eye-tracking datasets, using the `pymovements` library.

## Preparation steps
1. Download the data folder from the online repository. Note that this is only possible if you have access to at least one data collection protected folder. You will have access if you are an active member of one data collection group. Download the entire content of the folder.
When you download it from SwitchDrive, it will automatically create a .tar file.
2. Add the folder to the `data/` folder in this repo. The name of the folder is the data collection name, e.g., `MultiplEYE_ZH_CH_Zurich_1_2025`.
3. Extract the .tar file in the `data/` folder.
4. Make sure that the folder structure is correct. It should look like the one online and like this (there might be more data but this is not relevant at this point):
```
	MultiplEYE_ZH_CH_Zurich_1_2025/
		documentation/
		eye-tracking-sessions/
			001_.../
			002_.../
			...
			pilot_sessions/
				001_.../
				002_.../
				...
		psychometric-tests-sessions/
		stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/
		...
```

## The config file



The pipeline uses a config file which can be used to specify parameters and settings for the preprocessing. It is typically named `multipleye_settings_preprocessing.yaml`. You can load it explicitly or rely on the default loading mechanism (CWD, environment variable, or legacy root).

Once you have your config file ready, you can load it as shown below.

In [4]:
# from preprocessing.data_collection.multipleye_data_collection import prepare_language_folder
from preprocessing.data_collection.multipleye_data_collection import (
    MultipleyeDataCollection,
)

import preprocessing

# the settings will be loaded into general config module, so we can access all settings at the same place
from preprocessing import settings

from preprocessing.scripts.prepare_language_folder import prepare_language_folder

import polars as pl
from pathlib import Path

In [ ]:
# If you have a specific config file, load it here:
# settings.load_from_yaml("data/MultiplEYE_<...>/multipleye_settings_preprocessing.yaml")

In [5]:
# get the data collection name from the settings and create the path to the data folder
print(f"Active Data Collection: {settings.DATA_COLLECTION_NAME}")
print(f"Dataset Directory: {settings.DATASET_DIR}")

Active Data Collection: MultiplEYE_ZH_CH_Zurich_1_2025
Dataset Directory: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_ZH_CH_Zurich_1_2025


### Inspecting and overriding configuration

After loading the config, you can inspect which sessions are included or excluded, and override these values for the current session without modifying the YAML file.

In [6]:
print(f"Include pilots:  {settings.INCLUDE_PILOTS}")
print(f"Included:        {settings.INCLUDE_SESSIONS}")
print(f"Excluded:        {settings.EXCLUDE_SESSIONS}")
print(f"Output dir:      {settings.OUTPUT_DIR}")
print(f"Run preflight:   {settings.RUN_PREFLIGHT_CHECK}")
print(f"Recalculate:     {settings.RECALCULATE}")

# Override example (uncomment to limit processing to specific sessions):
# settings.INCLUDE_SESSIONS = ["014_DE_DE_1_ET1", "023_DE_DE_1_ET1"]
# settings.EXCLUDE_SESSIONS = []

Include pilots:  False
Included:        ['002_ZH_CH_1_ET1']
Excluded:        []
Output dir:      /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025
Run preflight:   True
Recalculate:     False


## MultiplEYE-specific preprocessing & cleaning

In order to be able to run a more generic preprocessing, the MultiplEYE data folder for one language needs to be cleaned and organized in a specific way. Running the script below will:
- unzip session folders if needed
- move session folders from core_sessions folder to the top folder
- check if there is a config file in the stimuli folder (if not, the stimulus folder was probably not uploaded correctly)
- check if there are psychometric tests (if applicable)
	- if necessary, restructure the psychometric test folder.

These steps are very individual for this data collection and results from bugs or changes across the years of collecting data.

Note that executing the cell below for the first time can take very long. However, it will run through quickly after this initial run.

In [7]:
preprocessed_stimulus_path = Path("/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025")

In [9]:
destination_aoi_path = Path("/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1")

In [41]:
aoi_files = list(destination_aoi_path.glob("[!.]*.csv"))

In [19]:
aoi_file = aoi_files[0]
aoi_question_file = aoi_files[1]

In [20]:
uoa_file = preprocessed_stimulus_path / "custom_units_of_analysis" / aoi_file.name

In [21]:
aoi_file

PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/arg_pisacowsmilk_10_aoi.csv')

In [60]:
uoa_column_name = "word"
uoa_idx_column_name = "word_idx"

In [22]:
uoa_data = pl.read_csv(uoa_file)[
            [uoa_column_name, uoa_idx_column_name, "char", "char_idx", "page",]
        ]

In [23]:
uoa_data

word,word_idx,char,char_idx,page
str,i64,str,i64,str
"""牛奶""",0,"""牛""",0,"""page_1"""
"""牛奶""",0,"""奶""",1,"""page_1"""
"""——""",1,"""—""",2,"""page_1"""
"""——""",1,"""—""",3,"""page_1"""
"""简介""",2,"""简""",4,"""page_1"""
…,…,…,…,…
"""“不”""",47,"""“""",87,"""page_13"""
"""“不”""",47,"""不""",88,"""page_13"""
"""“不”""",47,"""”""",89,"""page_13"""


In [26]:
uoa_data = uoa_data.sort(by=["page", "char_idx"])

In [27]:
uoa_data

word,word_idx,char,char_idx,page
str,i64,str,i64,str
"""牛奶""",0,"""牛""",0,"""page_1"""
"""牛奶""",0,"""奶""",1,"""page_1"""
"""——""",1,"""—""",2,"""page_1"""
"""——""",1,"""—""",3,"""page_1"""
"""简介""",2,"""简""",4,"""page_1"""
…,…,…,…,…
"""《大崩坏》""",1,"""崩""",37,"""page_9"""
"""《大崩坏》""",1,"""坏""",38,"""page_9"""
"""《大崩坏》""",1,"""》""",39,"""page_9"""


In [28]:
aoi_data = pl.read_csv(aoi_file)

In [29]:
aoi_data

char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,page,unit_of_analysis,unit_of_analysis_idx,unit_of_analysis_idx_in_line,secondary_unit_of_analysis_idx,secondary_unit_of_analysis_idx_in_line,secondary_unit_of_analysis
i64,str,f64,f64,i64,i64,i64,i64,str,str,i64,i64,i64,i64,str
0,"""牛""",81.0,88.0,28,34,0,0,"""page_1""","""牛奶""",0,1,0,0,"""牛奶——简介"""
1,"""奶""",109.0,88.0,28,34,1,0,"""page_1""","""牛奶""",0,1,0,0,"""牛奶——简介"""
2,"""—""",137.0,88.0,24,34,2,0,"""page_1""","""——""",1,2,0,0,"""牛奶——简介"""
3,"""—""",161.0,88.0,24,34,3,0,"""page_1""","""——""",1,2,0,0,"""牛奶——简介"""
4,"""简""",185.0,88.0,28,34,4,0,"""page_1""","""简介""",2,3,0,0,"""牛奶——简介"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
37,"""崩""",137.0,186.6,28,34,2,1,"""page_9""","""《大崩坏》""",1,1,1,0,"""《大崩坏》书评"""
38,"""坏""",165.0,186.6,28,34,3,1,"""page_9""","""《大崩坏》""",1,1,1,0,"""《大崩坏》书评"""
39,"""》""",193.0,186.6,28,34,4,1,"""page_9""","""《大崩坏》""",1,1,1,0,"""《大崩坏》书评"""


In [30]:
aoi_data.sort(by=["page", "char_idx"])

char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,page,unit_of_analysis,unit_of_analysis_idx,unit_of_analysis_idx_in_line,secondary_unit_of_analysis_idx,secondary_unit_of_analysis_idx_in_line,secondary_unit_of_analysis
i64,str,f64,f64,i64,i64,i64,i64,str,str,i64,i64,i64,i64,str
0,"""牛""",81.0,88.0,28,34,0,0,"""page_1""","""牛奶""",0,1,0,0,"""牛奶——简介"""
1,"""奶""",109.0,88.0,28,34,1,0,"""page_1""","""牛奶""",0,1,0,0,"""牛奶——简介"""
2,"""—""",137.0,88.0,24,34,2,0,"""page_1""","""——""",1,2,0,0,"""牛奶——简介"""
3,"""—""",161.0,88.0,24,34,3,0,"""page_1""","""——""",1,2,0,0,"""牛奶——简介"""
4,"""简""",185.0,88.0,28,34,4,0,"""page_1""","""简介""",2,3,0,0,"""牛奶——简介"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
37,"""崩""",137.0,186.6,28,34,2,1,"""page_9""","""《大崩坏》""",1,1,1,0,"""《大崩坏》书评"""
38,"""坏""",165.0,186.6,28,34,3,1,"""page_9""","""《大崩坏》""",1,1,1,0,"""《大崩坏》书评"""
39,"""》""",193.0,186.6,28,34,4,1,"""page_9""","""《大崩坏》""",1,1,1,0,"""《大崩坏》书评"""


In [73]:
uoa_chars = list(uoa_data["char"])

In [74]:
uoa_chars

['月',
 '相',
 '的',
 '成',
 '因',
 '是',
 '什',
 '么',
 '？',
 '月',
 '球',
 '自',
 '转',
 '周',
 '期',
 '与',
 '绕',
 '地',
 '球',
 '一',
 '周',
 '的',
 '轨',
 '道',
 '周',
 '期',
 '相',
 '等',
 '。',
 '月',
 '球',
 '自',
 '转',
 '过',
 '程',
 '中',
 '潮',
 '汐',
 '倾',
 '斜',
 '的',
 '位',
 '置',
 '。',
 '月',
 '球',
 '的',
 '质',
 '量',
 '和',
 '表',
 '面',
 '重',
 '力',
 '。',
 '太',
 '阳',
 '系',
 '的',
 '自',
 '转',
 '。',
 '为',
 '什',
 '么',
 '月',
 '球',
 '对',
 '地',
 '球',
 '如',
 '此',
 '重',
 '要',
 '？',
 '因',
 '为',
 '它',
 '是',
 '地',
 '球',
 '唯',
 '一',
 '的',
 '卫',
 '星',
 '并',
 '且',
 '施',
 '加',
 '引',
 '力',
 '。',
 '由',
 '于',
 '其',
 '质',
 '量',
 '和',
 '表',
 '面',
 '重',
 '力',
 '较',
 '大',
 '。',
 '因',
 '为',
 '从',
 '地',
 '球',
 '上',
 '可',
 '以',
 '看',
 '到',
 '它',
 '。',
 '因',
 '为',
 '它',
 '是',
 '一',
 '颗',
 '天',
 '然',
 '卫',
 '星',
 '。']

In [90]:
aoi_data = pl.read_csv(aoi_question_file)

In [91]:
aoi_data = aoi_data.sort(by=["page", "line_idx", "char_idx"])

In [92]:
aoi_data.filter(aoi_data["question_image_version"] == "question_images_version_1")

char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,page,unit_of_analysis_idx,unit_of_analysis_idx_in_line,unit_of_analysis,question_image_version
i64,str,f64,f64,i64,i64,i64,i64,str,i64,i64,str,str
0,"""月""",196.5,247.75,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_1"""
1,"""球""",224.5,247.75,28,34,1,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_1"""
2,"""自""",252.5,247.75,28,34,2,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_1"""
3,"""转""",280.5,247.75,28,34,3,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_1"""
4,"""过""",308.5,247.75,28,34,4,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_1"""
…,…,…,…,…,…,…,…,…,…,…,…,…
8,"""如""",305.0,88.0,28,34,8,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_1"""
9,"""此""",333.0,88.0,29,34,9,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_1"""
10,"""重""",362.0,88.0,28,34,10,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_1"""


In [88]:
aoi_question_file

PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/enc_wikimoon_13_aoi_questions.csv')

In [112]:
uoa_file

PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/custom_units_of_analysis/enc_wikimoon_13_aoi_questions.csv')

In [116]:
uoa_data = uoa_data.sort(by=["page", "char_idx"])

In [117]:
uoa_data

segment,segment_idx,char,char_idx,page,question_image_version
str,i64,str,i64,str,str
"""月球""",16,"""月""",0,"""Enc_WikiMoon_13_question_13121…","""question_images_version_1"""
"""月球""",16,"""球""",1,"""Enc_WikiMoon_13_question_13121…","""question_images_version_1"""
"""自传""",17,"""自""",2,"""Enc_WikiMoon_13_question_13121…","""question_images_version_1"""
"""自传""",17,"""转""",3,"""Enc_WikiMoon_13_question_13121…","""question_images_version_1"""
"""过程""",18,"""过""",4,"""Enc_WikiMoon_13_question_13121…","""question_images_version_1"""
…,…,…,…,…,…
"""如此""",4,"""如""",8,"""question_13131""","""question_images_version_1"""
"""如此""",4,"""此""",9,"""question_13131""","""question_images_version_1"""
"""重要？""",5,"""重""",10,"""question_13131""","""question_images_version_1"""


In [96]:
uoa_chars = list(uoa_data["char"])

In [118]:
aoi_chars = list(aoi_data["char"].filter(aoi_data["question_image_version"] == "question_images_version_1"))

In [98]:
aoi_chars

['月',
 '球',
 '自',
 '转',
 '过',
 '程',
 '中',
 '潮',
 '汐',
 '倾',
 '斜',
 '的',
 '位',
 '置',
 '。',
 '月',
 '球',
 '的',
 '质',
 '量',
 '和',
 '表',
 '面',
 '重',
 '力',
 '。',
 '太',
 '阳',
 '系',
 '的',
 '自',
 '转',
 '。',
 '月',
 '球',
 '自',
 '转',
 '周',
 '期',
 '与',
 '绕',
 '地',
 '球',
 '一',
 '周',
 '的',
 '轨',
 '道',
 '周',
 '期',
 '相',
 '等',
 '。',
 '由',
 '于',
 '其',
 '质',
 '量',
 '和',
 '表',
 '面',
 '重',
 '力',
 '较',
 '大',
 '。',
 '因',
 '为',
 '从',
 '地',
 '球',
 '上',
 '可',
 '以',
 '看',
 '到',
 '它',
 '。',
 '因',
 '为',
 '它',
 '是',
 '一',
 '颗',
 '天',
 '然',
 '卫',
 '星',
 '。',
 '因',
 '为',
 '它',
 '是',
 '地',
 '球',
 '唯',
 '一',
 '的',
 '卫',
 '星',
 '并',
 '且',
 '施',
 '加',
 '引',
 '力',
 '。',
 '月',
 '相',
 '的',
 '成',
 '因',
 '是',
 '什',
 '么',
 '？',
 '为',
 '什',
 '么',
 '月',
 '球',
 '对',
 '地',
 '球',
 '如',
 '此',
 '重',
 '要',
 '？']

In [75]:
aoi_chars = list(aoi_data["char"])
short_aoi_chars = aoi_chars[:-4]

In [119]:
len(uoa_chars) == len(aoi_chars)

True

In [100]:
uoa_chars

['月',
 '球',
 '自',
 '转',
 '过',
 '程',
 '中',
 '潮',
 '汐',
 '倾',
 '斜',
 '的',
 '位',
 '置',
 '。',
 '月',
 '球',
 '的',
 '质',
 '量',
 '和',
 '表',
 '面',
 '重',
 '力',
 '。',
 '太',
 '阳',
 '系',
 '的',
 '自',
 '转',
 '。',
 '月',
 '球',
 '自',
 '转',
 '周',
 '期',
 '与',
 '绕',
 '地',
 '球',
 '一',
 '周',
 '的',
 '轨',
 '道',
 '周',
 '期',
 '相',
 '等',
 '。',
 '由',
 '于',
 '其',
 '质',
 '量',
 '和',
 '表',
 '面',
 '重',
 '力',
 '较',
 '大',
 '。',
 '因',
 '为',
 '从',
 '地',
 '球',
 '上',
 '可',
 '以',
 '看',
 '到',
 '它',
 '。',
 '因',
 '为',
 '它',
 '是',
 '一',
 '颗',
 '天',
 '然',
 '卫',
 '星',
 '。',
 '因',
 '为',
 '它',
 '是',
 '地',
 '球',
 '唯',
 '一',
 '的',
 '卫',
 '星',
 '并',
 '且',
 '施',
 '加',
 '引',
 '力',
 '。',
 '月',
 '相',
 '的',
 '成',
 '因',
 '是',
 '什',
 '么',
 '？',
 '为',
 '什',
 '么',
 '月',
 '球',
 '对',
 '地',
 '球',
 '如',
 '此',
 '重',
 '要',
 '？']

In [84]:
aoi_chars

['月',
 '球',
 '自',
 '转',
 '过',
 '程',
 '中',
 '潮',
 '汐',
 '倾',
 '斜',
 '的',
 '位',
 '置',
 '。',
 '月',
 '球',
 '的',
 '质',
 '量',
 '和',
 '表',
 '面',
 '重',
 '力',
 '。',
 '太',
 '阳',
 '系',
 '的',
 '自',
 '转',
 '。',
 '月',
 '球',
 '自',
 '转',
 '周',
 '期',
 '与',
 '绕',
 '地',
 '球',
 '一',
 '周',
 '的',
 '轨',
 '道',
 '周',
 '期',
 '相',
 '等',
 '。',
 '由',
 '于',
 '其',
 '质',
 '量',
 '和',
 '表',
 '面',
 '重',
 '力',
 '较',
 '大',
 '。',
 '因',
 '为',
 '从',
 '地',
 '球',
 '上',
 '可',
 '以',
 '看',
 '到',
 '它',
 '。',
 '因',
 '为',
 '它',
 '是',
 '一',
 '颗',
 '天',
 '然',
 '卫',
 '星',
 '。',
 '因',
 '为',
 '它',
 '是',
 '地',
 '球',
 '唯',
 '一',
 '的',
 '卫',
 '星',
 '并',
 '且',
 '施',
 '加',
 '引',
 '力',
 '。',
 '月',
 '相',
 '的',
 '成',
 '因',
 '是',
 '什',
 '么',
 '？',
 '为',
 '什',
 '么',
 '月',
 '球',
 '对',
 '地',
 '球',
 '如',
 '此',
 '重',
 '要',
 '？']

In [15]:
import difflib

In [83]:
for line in difflib.unified_diff(aoi_chars, uoa_chars, lineterm=""):
            print(line)

--- 
+++ 
@@ -1,3 +1,32 @@
+月
+相
+的
+成
+因
+是
+什
+么
+？
+月
+球
+自
+转
+周
+期
+与
+绕
+地
+球
+一
+周
+的
+轨
+道
+周
+期
+相
+等
+。
 月
 球
 自
@@ -31,25 +60,36 @@
 自
 转
 。
+为
+什
+么
 月
 球
-自
-转
-周
-期
-与
-绕
+对
 地
 球
+如
+此
+重
+要
+？
+因
+为
+它
+是
+地
+球
+唯
 一
-周
 的
-轨
-道
-周
-期
-相
-等
+卫
+星
+并
+且
+施
+加
+引
+力
 。
 由
 于
@@ -87,43 +127,3 @@
 卫
 星
 。
-因
-为
-它
-是
-地
-球
-唯
-一
-的
-卫
-星
-并
-且
-施
-加
-引
-力
-。
-月
-相
-的
-成
-因
-是
-什
-么
-？
-为
-什
-么
-月
-球
-对
-地
-球
-如
-此
-重
-要
-？


In [101]:
aoi_chars == uoa_chars

True

In [124]:
new_aois = aoi_data.join(uoa_data, on=["page", "char_idx", "char"])

In [122]:
aoi_data

char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,page,secondary_unit_of_analysis_idx,secondary_unit_of_analysis_idx_in_line,secondary_unit_of_analysis,question_image_version
i64,str,f64,f64,i64,i64,i64,i64,str,i64,i64,str,str
0,"""月""",196.5,247.75,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_25"""
0,"""月""",694.3,436.04,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_50"""
0,"""月""",196.5,743.25,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_99"""
0,"""月""",196.5,247.75,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_249"""
0,"""月""",196.5,743.25,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_225"""
…,…,…,…,…,…,…,…,…,…,…,…,…
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_56"""
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_151"""
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_226"""


In [121]:
aoi_data = aoi_data.rename(
        {
            "unit_of_analysis_idx": "secondary_unit_of_analysis_idx",
            "unit_of_analysis": "secondary_unit_of_analysis",
            "unit_of_analysis_idx_in_line": "secondary_unit_of_analysis_idx_in_line",})

ColumnNotFoundError: "unit_of_analysis_idx" not found

In [ ]:
def _add_custom_uoa(
    stimuli_folder: Path, aoi_file: Path, uoa_column_name, uoa_idx_column_name, custom_file_name: str = ""
) -> None:
    """
    The custom units of analysis can be added as follows for ZH:
    - They are in a directory called "custom_units_of_analysis" in the data folder
    - The aoi files for the questions are named and structured the same as the original question aoi files.
        - They contain an additional segment and segment_idx column
        - As the text on all pages is for all question versions is the same, only question_images_version_1 is annotated
    - The stimuli text aoi files are name like this: Lit_Alchemist_zh_seg_pages.csv. The first part can be replaced
    by the respective stimulus name
        - They contain an additional word and word_idx column

    How to add it:
    - Merge the custom and the original aoi files on the char and char index as well as the page column!
    - check that all chars listed in both files exactly align. Index and char should be exactly the same.
    - ensure the segment contains the actual characters that it annotates, and that the running segment index is correct
    - for the questions add the segments to all question images versions and make sure the characters match. Only the coordinates should be different for the options

    Output:
    The original aoi files should be exactly the same as before also the file name, but they should contain two additional columns:
        - unit_of_analysis
        - unit_of_analysis_idx

    If there is a word column already in there: rename to secondary_unit_of_analysis and word_idx to secondary_unit_of_analysis_idx.
    The characters and character indices should stay exactly the same.

    If any of the characters and segment characters do not match, it will be reported.

    """

    aoi_data = pl.read_csv(aoi_file)
    if "secondary_unit_of_analysis" in aoi_data:
        # If this file was already merged
        # If the costum units of analysis change, the stimulus folder is automatically copied again and merged
        return

    output_path = stimuli_folder / "merge_output" / aoi_file.name
    aoi_data = aoi_data.sort(by=["page", "line_idx", "char_idx"])
    
    
    uoa_file = stimuli_folder / "custom_units_of_analysis" / aoi_file.name

    if not uoa_file.exists():
        #TODO What about files that are just missing e.g. the question files for Greenlandic
        raise FileNotFoundError(
            f"We could not find a matching uoa file for {aoi_file.name}. Please make "
            f"sure that the uoa file is named exactly as the corresponding aoi file and "
            f"that it is in your data folder in a folder called 'custom_units_of_analysis'."
        )

    if "questions" in uoa_file.name:
        uoa_data = pl.read_csv(uoa_file)[
            [
                uoa_column_name,
                uoa_idx_column_name,
                "char",
                "char_idx",
                "page",
                "question_image_version",
            ]
        ]
        uoa_data = uoa_data.filter(
            uoa_data["question_image_version"] == "question_images_version_1"
        )

        uoa_data = uoa_data[
            [uoa_column_name, uoa_idx_column_name, "char", "char_idx", "page"]
        ]

    else:
        uoa_data = pl.read_csv(uoa_file)[
            [uoa_column_name, uoa_idx_column_name, "char", "char_idx", "page"]
        ]

    uoa_data = uoa_data.sort(by=["page", "char_idx"])
    uoa_chars = list(uoa_data["char"])


    if "questions" in uoa_file.name:
        aoi_chars = list(aoi_data["char"].filter(
            aoi_data["question_image_version"] == "question_images_version_1"))
    else:
        aoi_chars = list(aoi_data["char"])


    # check that the char columns match exactly
    if not aoi_chars == uoa_chars:
        print(f"The custom units of analysis file for the corresponding aoi file {aoi_file.name}, contain different characters. Please verify that the stimuli in both files match exactly.")
        print(f"Using default whitespace delimited units of analysis for {aoi_file.name}")
        return

    aoi_data = aoi_data.rename(
        {
            "unit_of_analysis_idx": "secondary_unit_of_analysis_idx",
            "unit_of_analysis": "secondary_unit_of_analysis",
            "unit_of_analysis_idx_in_line": "secondary_unit_of_analysis_idx_in_line",            
        }
    )

    uoa_data = uoa_data.rename(
        {
            uoa_column_name: "unit_of_analysis",
            uoa_idx_column_name: "unit_of_analysis_idx",
        }
    )
    
    
    # join the segment and segment idx on page and char idx
    new_aois = aoi_data.join(uoa_data, on=["page", "char_idx", "char"])


    # get the line index of the segments
    new_aois = new_aois.with_columns(
        pl.col("unit_of_analysis_idx")
        .rank(method="dense")
        .over(["page", "line_idx"])
        .cast(pl.Int64)
        .alias("unit_of_analysis_idx_in_line")
    )

    col_order = [
        "char_idx",
        "char",
        "top_left_x",
        "top_left_y",
        "width",
        "height",
        "char_idx_in_line",
        "line_idx",
        "page",
        "unit_of_analysis_idx",
        "unit_of_analysis_idx_in_line",
        "unit_of_analysis",
        "secondary_unit_of_analysis_idx",
        "secondary_unit_of_analysis_idx_in_line",
        "secondary_unit_of_analysis",
    ]

    new_aois = new_aois[col_order]
    new_aois = new_aois.sort(by=["page", "line_idx", "char_idx"])
    new_aois.write_csv(output_path)

In [13]:
aoi_files

[PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/arg_pisacowsmilk_10_aoi.csv'),
 PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/enc_wikimoon_13_aoi_questions.csv'),
 PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/lit_solaris_8_aoi.csv'),
 PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/lit_northwind_7_aoi.csv'),
 PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/

In [45]:
"secondary_unit_of_analysis" in aoi_data

True

In [50]:
aoi_data.drop(["char_idx", "char", "bleh"])

ColumnNotFoundError: "bleh" not found

In [48]:
aoi_data["char_idx", "char"]

char_idx,char
i64,str
0,"""牛"""
1,"""奶"""
2,"""—"""
3,"""—"""
4,"""简"""
…,…
37,"""崩"""
38,"""坏"""
39,"""》"""


In [43]:
for aoi_file in aoi_files:
    if "question" in aoi_file.name:
        uoa_column_name = "segment"
        uoa_idx_column_name = "segment_idx"
    else:
        uoa_column_name = "word"
        uoa_idx_column_name = "word_idx"

    _add_custom_uoa(stimuli_folder=preprocessed_stimulus_path, aoi_file=aoi_file, uoa_column_name=uoa_column_name, uoa_idx_column_name=uoa_idx_column_name)

encountered error
aoi_chars
1467
['牛', '奶', '—', '—', '简', '介', '在', '有', '些', '地', '方', '，', '像', '美', '国', '，', '喝', '牛', '奶', '是', '件', '很', '平', '常', '的', '事', '。', '试', '想', '有', '三', '位', '学', '生', '，', '分', '别', '是', '安', '娜', '、', '克', '里', '斯', '托', '弗', '和', '山', '姆', '，', '正', '在', '美', '国', '的', '一', '家', '咖', '啡', '店', '里', '。', '老', '板', '最', '近', '在', '橱', '窗', '中', '放', '置', '了', '一', '块', '招', '牌', '，', '上', '面', '写', '着', '：', '“', '4', '月', '5', '日', '起', '本', '店', '将', '不', '再', '供', '应', '牛', '奶', '，', '改', '用', '供', '应', '豆', '奶', '替', '代', '。', '”', '安', '娜', '、', '克', '里', '斯', '托', '弗', '和', '山', '姆', '很', '想', '知', '道', '咖', '啡', '店', '为', '什', '么', '要', '停', '止', '供', '应', '牛', '奶', '，', '于', '是', '安', '娜', '用', '她', '的', '智', '能', '手', '机', '上', '网', '搜', '索', '“', '牛', '奶', '”', '。', '他', '们', '看', '了', '第', '一', '条', '搜', '索', '结', '果', '并', '进', '行', '讨', '论', '。', '多', '年', '来', '，', '美', '国', '农', '业', '部', '、', '美', '国', '乳', '制', '品', '委', '员', '会', '

In [40]:
aoi_data

char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,page,secondary_unit_of_analysis_idx,secondary_unit_of_analysis_idx_in_line,secondary_unit_of_analysis
i64,str,f64,f64,i64,i64,i64,i64,str,i64,i64,str
0,"""牛""",81.0,88.0,28,34,0,0,"""page_1""",0,0,"""牛奶——简介"""
1,"""奶""",109.0,88.0,28,34,1,0,"""page_1""",0,0,"""牛奶——简介"""
2,"""—""",137.0,88.0,24,34,2,0,"""page_1""",0,0,"""牛奶——简介"""
3,"""—""",161.0,88.0,24,34,3,0,"""page_1""",0,0,"""牛奶——简介"""
4,"""简""",185.0,88.0,28,34,4,0,"""page_1""",0,0,"""牛奶——简介"""
…,…,…,…,…,…,…,…,…,…,…,…
37,"""崩""",137.0,186.6,28,34,2,1,"""page_9""",1,0,"""《大崩坏》书评"""
38,"""坏""",165.0,186.6,28,34,3,1,"""page_9""",1,0,"""《大崩坏》书评"""
39,"""》""",193.0,186.6,28,34,4,1,"""page_9""",1,0,"""《大崩坏》书评"""


In [111]:
uoa_column_name = "segment"
uoa_idx_column_name = "segment_idx"

In [123]:
uoa_data = uoa_data.rename({uoa_column_name: "unit_of_analysis",
            uoa_idx_column_name: "unit_of_analysis_idx",})

In [110]:
uoa_data

unit_of_analysis,unit_of_analysis_idx,char,char_idx,page
str,i64,str,i64,str
"""月球自转过程中潮汐倾斜的位置。""",0,"""月""",0,"""Enc_WikiMoon_13_question_13121…"
"""月球自转过程中潮汐倾斜的位置。""",0,"""球""",1,"""Enc_WikiMoon_13_question_13121…"
"""月球自转过程中潮汐倾斜的位置。""",0,"""自""",2,"""Enc_WikiMoon_13_question_13121…"
"""月球自转过程中潮汐倾斜的位置。""",0,"""转""",3,"""Enc_WikiMoon_13_question_13121…"
"""月球自转过程中潮汐倾斜的位置。""",0,"""过""",4,"""Enc_WikiMoon_13_question_13121…"
…,…,…,…,…
"""为什么月球对地球如此重要？""",0,"""如""",8,"""question_13131"""
"""为什么月球对地球如此重要？""",0,"""此""",9,"""question_13131"""
"""为什么月球对地球如此重要？""",0,"""重""",10,"""question_13131"""


In [109]:
new_aois = aoi_data.join(uoa_data, on=["page", "char_idx", "char"])
new_aois

char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,page,secondary_unit_of_analysis_idx,secondary_unit_of_analysis_idx_in_line,secondary_unit_of_analysis,question_image_version,unit_of_analysis,unit_of_analysis_idx
i64,str,f64,f64,i64,i64,i64,i64,str,i64,i64,str,str,str,i64
0,"""月""",196.5,247.75,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_25""","""月球自转过程中潮汐倾斜的位置。""",0
0,"""月""",694.3,436.04,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_50""","""月球自转过程中潮汐倾斜的位置。""",0
0,"""月""",196.5,743.25,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_99""","""月球自转过程中潮汐倾斜的位置。""",0
0,"""月""",196.5,247.75,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_249""","""月球自转过程中潮汐倾斜的位置。""",0
0,"""月""",196.5,743.25,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_225""","""月球自转过程中潮汐倾斜的位置。""",0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_56""","""为什么月球对地球如此重要？""",0
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_151""","""为什么月球对地球如此重要？""",0
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_226""","""为什么月球对地球如此重要？""",0


In [125]:
new_aois = new_aois.with_columns(
    pl.col("unit_of_analysis_idx")
    .rank(method="dense")
    .over(["page", "line_idx"])
    .cast(pl.Int64)
    .alias("unit_of_analysis_idx_in_line")
)

In [126]:
new_aois

char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,page,secondary_unit_of_analysis_idx,secondary_unit_of_analysis_idx_in_line,secondary_unit_of_analysis,question_image_version,unit_of_analysis,unit_of_analysis_idx,question_image_version_right,unit_of_analysis_idx_in_line
i64,str,f64,f64,i64,i64,i64,i64,str,i64,i64,str,str,str,i64,str,i64
0,"""月""",196.5,247.75,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_25""","""月球""",16,"""question_images_version_1""",1
0,"""月""",694.3,436.04,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_50""","""月球""",16,"""question_images_version_1""",1
0,"""月""",196.5,743.25,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_99""","""月球""",16,"""question_images_version_1""",1
0,"""月""",196.5,247.75,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_249""","""月球""",16,"""question_images_version_1""",1
0,"""月""",196.5,743.25,28,34,0,0,"""Enc_WikiMoon_13_question_13121…",0,0,"""月球自转过程中潮汐倾斜的位置。""","""question_images_version_225""","""月球""",16,"""question_images_version_1""",1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_56""","""重要？""",5,"""question_images_version_1""",6
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_151""","""重要？""",5,"""question_images_version_1""",6
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""",0,0,"""为什么月球对地球如此重要？""","""question_images_version_226""","""重要？""",5,"""question_images_version_1""",6


In [127]:
col_order = [
        "char_idx",
        "char",
        "top_left_x",
        "top_left_y",
        "width",
        "height",
        "char_idx_in_line",
        "line_idx",
        "page",
        "unit_of_analysis",
        "unit_of_analysis_idx",
        "unit_of_analysis_idx_in_line",
        "secondary_unit_of_analysis_idx",
        "secondary_unit_of_analysis_idx_in_line",
        "secondary_unit_of_analysis",
    ]

In [128]:
new_aois = new_aois[col_order]

In [129]:
new_aois

char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,page,unit_of_analysis,unit_of_analysis_idx,unit_of_analysis_idx_in_line,secondary_unit_of_analysis_idx,secondary_unit_of_analysis_idx_in_line,secondary_unit_of_analysis
i64,str,f64,f64,i64,i64,i64,i64,str,str,i64,i64,i64,i64,str
0,"""月""",196.5,247.75,28,34,0,0,"""Enc_WikiMoon_13_question_13121…","""月球""",16,1,0,0,"""月球自转过程中潮汐倾斜的位置。"""
0,"""月""",694.3,436.04,28,34,0,0,"""Enc_WikiMoon_13_question_13121…","""月球""",16,1,0,0,"""月球自转过程中潮汐倾斜的位置。"""
0,"""月""",196.5,743.25,28,34,0,0,"""Enc_WikiMoon_13_question_13121…","""月球""",16,1,0,0,"""月球自转过程中潮汐倾斜的位置。"""
0,"""月""",196.5,247.75,28,34,0,0,"""Enc_WikiMoon_13_question_13121…","""月球""",16,1,0,0,"""月球自转过程中潮汐倾斜的位置。"""
0,"""月""",196.5,743.25,28,34,0,0,"""Enc_WikiMoon_13_question_13121…","""月球""",16,1,0,0,"""月球自转过程中潮汐倾斜的位置。"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""","""重要？""",5,6,0,0,"""为什么月球对地球如此重要？"""
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""","""重要？""",5,6,0,0,"""为什么月球对地球如此重要？"""
12,"""？""",418.0,88.0,28,34,12,0,"""question_13131""","""重要？""",5,6,0,0,"""为什么月球对地球如此重要？"""


In [105]:
new_aois.columns

['char_idx',
 'char',
 'top_left_x',
 'top_left_y',
 'width',
 'height',
 'char_idx_in_line',
 'line_idx',
 'page',
 'unit_of_analysis_idx',
 'unit_of_analysis_idx_in_line',
 'unit_of_analysis',
 'question_image_version',
 'word',
 'word_idx']

In [54]:
new_aois = new_aois.sort(by=["page", "line_idx", "char_idx"])

In [57]:
new_aois.write_csv(aoi_file)

In [58]:
aoi_file

PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/arg_pisacowsmilk_10_aoi.csv')

In [4]:
# run the preparation function to prepare the language folder structure
prepare_language_folder()

2026-10-30 12:51:06,286 - preprocessing.scripts.prepare_language_folder - WARNING - The 'psychometric-tests-sessions' folder does not exist in '/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_ZH_CH_Zurich_1_2025'. If this data collection includes psychometric tests, create the folder and unpack the data in the cleaning step.
Expected structure after unpacking (task-first):
  psychometric-tests-sessions/
    psychometric_test_ZH_CH_1/
      PLAB/{sid}/...
      RAN/{sid}/...
      Stroop_Flanker/{sid}/...
      WMC/{sid}/...
      WikiVocab/{sid}/...
    participant_configs_ZH_CH_1/
      {sid}.yaml
Skipping psychometric tests preparation.
2026-10-30 12:51:06,894 - preprocessing - INFO - Copying stimulus assets to /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025...
2026-10-30 12:51:07,327 - preprocessing - INFO - Copying question_images_version_228...
2026-10-30 12:51:0

Next, we create a `MultipleyeDataCollection` object from the data folder. This will allow us to easily access the sessions and their information in the next steps.

In [7]:
multipleye = MultipleyeDataCollection.create_from_data_folder(
    settings.DATASET_DIR,
    include_pilots=settings.INCLUDE_PILOTS,
    excluded_sessions=settings.EXCLUDE_SESSIONS,
    included_sessions=settings.INCLUDE_SESSIONS,
)

2026-10-30 13:03:01,693 - py.warnings - PYWARN:WARNING - preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/config/config_zh_ch_Zurich_1_2025.py:9: SyntaxWarning: invalid escape sequence '\m'
  question_images_question_images_version_1_csv = "data/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/question_images_zh_ch_1/question_images_version_1\multipleye_comprehension_questions_zh_question_images_version_1_with_img_paths.csv"

2026-10-30 13:03:01,695 - py.warnings - PYWARN:WARNING - preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/config/config_zh_ch_Zurich_1_2025.py:10: SyntaxWarning: invalid escape sequence '\m'
  question_images_question_images_version_2_csv = "data/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/question_images_zh_ch_1/question_images_version_2\multipleye_comprehension_questions_zh_question_images_version_2_with_img_paths.csv"

2026-10-30 13:03:01,698 - py.warnings - PYWARN:WARNING - preprocessed_data/Multip

In [5]:
from preprocessing.utils.stimulus_versions import (
    configured_version_suffixes,
    default_version_suffix,
    stimulus_folder_name,
    version_suffix_for_pid,
)

In [10]:
multipleye.data_collection_name

'MultiplEYE_ZH_CH_Zurich_1_2025'

In [20]:
folder_name = stimulus_folder_name(multipleye.data_collection_name, suffix = None)

In [21]:
preprocessed_stimulus_path = settings.OUTPUT_DIR / folder_name

In [22]:
destination_aoi_path = (
            preprocessed_stimulus_path
            / f"aoi_stimuli_{multipleye.language.lower()}_{multipleye.country.lower()}_{multipleye.lab_number}"
        )

In [23]:
destination_aoi_path

PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1')

In [24]:
aoi_files = list(destination_aoi_path.glob("*.csv"))

In [25]:
aoi_files

[PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/arg_pisacowsmilk_10_aoi.csv'),
 PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/enc_wikimoon_13_aoi_questions.csv'),
 PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/lit_solaris_8_aoi.csv'),
 PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/lit_northwind_7_aoi.csv'),
 PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/

In [49]:
test_file = aoi_files[1]

In [50]:
test_file

PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_ZH_CH_Zurich_1_2025/stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/aoi_stimuli_zh_ch_1/enc_wikimoon_13_aoi_questions.csv')

In [51]:
def rename_aoi_columns(aoi_file: Path) -> None:
    """
    Refactor the aoi files for a given data collection to have consistent naming conventions.
    If necessary, add the custom uoa aois for the given language.
    :param aoi_file: Path to the AOI file
    """

    aoi_data = pl.read_csv(aoi_file)

    # rename columns
    try:
        aoi_data = aoi_data.rename(
            {
                "word_idx": "unit_of_analysis_idx",
                "word": "unit_of_analysis",
                "word_idx_in_line": "unit_of_analysis_idx_in_line",
            },
            strict=False
        )
    except pl.exceptions.ColumnNotFoundError:
        return

    # save the refactored aoi files
    aoi_data = aoi_data.sort(by=["page", "char_idx"])
    aoi_data.write_csv(aoi_file)


In [53]:
for aoi_file in aoi_files:
    rename_aoi_columns(aoi_file)

### Preflight check

Before processing, run a preflight check to validate the dataset structure and catch common issues (missing files, incorrect folder layout, etc.). In case EDF files are missing, you can use `settings.EXCLUDE_SESSIONS = []` to exclude specific sessions, as shown a few cells above.

In [16]:
preprocessing.run_preflight_check(multipleye)

PreflightError: 
========================================================
  Preflight check FAILED — 2 error(s)
========================================================

## Stage 0: Converting EDF to ASC and Preparing Session-Level Information

Stage 0 refers to the initial steps of preprocessing, which involve converting raw eye-tracking data from its original format (e.g., EDF) into a more accessible format (e.g., ASC), and preparing session-level information. This stage is specific to EyeLink eye-trackers and can be omitted for other eye-trackers.

In [54]:
multipleye.convert_edf_to_asc()

2026-10-30 14:24:05,662 - preprocessing - INFO - Starting EDF to ASC conversion for 1 sessions.
2026-10-30 14:24:05,668 - preprocessing - INFO - EDF to ASC conversion completed.


Once this conversion has been completed, we can load all sessions and parse the .asc files.

In [55]:
multipleye.prepare_session_level_information()

Preparing session 002_ZH_CH_1_ET1: 100%|██████████| 1/1 [00:00<00:00,  4.90it/s]
0it [00:00, ?it/s]


In [56]:
# print an overview on the data collection and the sessions
multipleye

[administrative]
  title: MultiplEYE_ZH_CH_Zurich_1_2025
  dataset_type: MultiplEYE
  dataset_description: CH, Zurich (lab 1). ZH corpus. Data collected from unknown to unknown.
  number_of_sessions: 1
  number_of_pilots: 0
  number_of_et_sessions_per_participant: 1
  session_completeness: {'expected_sessions_per_participant': 1, 'total_participants': 1, 'complete_participants': 1}
  city: Zurich
  lab_number: 1
  country: CH
  tested_language: ZH
[language_details]
  metadata_form_exists: False
  language_script: None
  language_family: None
  start_date_of_data_collection: None
  end_date_of_data_collection: None
[data_availability]
  raw_data_available: True
  fixations_available: True
  saccades_available: True
  reading_measures_available: True
[psychometric_tests]
  tests_available: ['LWMC', 'PLAB', 'Flanker', 'Stroop', 'RAN', 'WikiVocab']
[technical_setup]
  eye_tracker_name_raw: EyeLink Portable Duo
  eye_tracker_name: EyeLink Portable Duo
  sampling_frequency_hz: 1000
  monito

## Stage 1: Extracting Gaze Samples

In the first preprocessing stage, we extract gaze samples from the .asc files and create a gaze dataframe for each session. This dataframe contains the raw gaze data, including the x and y coordinates of the gaze, the timestamp. We also save the raw gaze data in a separate file for each session.

The next steps are performed for one session only. It is always possible to loop over all sessions and apply the same preprocessing steps to each of them, but for the sake of clarity and simplicity, we will work with one session as an example.



In [57]:
# pick only one session as an example to work with in the next steps
sessions = list(multipleye)  # list of Session objects
sess = sessions[0]  # a real Session
sid = sess.sid  # get the session ID (Sid) from the Session
sid

Sid(pid='002', lang='ZH', country='CH', lab='1', session='ET1', session_id=1, postfix='')

In [58]:
sess.stimuli[0].text_stimulus

char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,page,unit_of_analysis_idx,unit_of_analysis_idx_in_line,unit_of_analysis
i64,str,f64,f64,i64,f64,i64,i64,str,i64,i64,str
0,"""魔""",81.0,88.0,28,34.0,0,0,"""page_1""",0,0,"""魔山——前言"""
1,"""山""",109.0,88.0,28,34.0,1,0,"""page_1""",0,0,"""魔山——前言"""
2,"""—""",137.0,88.0,24,34.0,2,0,"""page_1""",0,0,"""魔山——前言"""
3,"""—""",161.0,88.0,24,34.0,3,0,"""page_1""",0,0,"""魔山——前言"""
4,"""前""",185.0,88.0,28,34.0,4,0,"""page_1""",0,0,"""魔山——前言"""
…,…,…,…,…,…,…,…,…,…,…,…
81,"""花""",109.0,252.9,28,98.6,1,2,"""page_6""",2,0,"""要花七年哩！"""
82,"""七""",137.0,252.9,28,98.6,2,2,"""page_6""",2,0,"""要花七年哩！"""
83,"""年""",165.0,252.9,28,98.6,3,2,"""page_6""",2,0,"""要花七年哩！"""


In [11]:
type(sess)

preprocessing.data_collection.session.Session

### Creating Gaze Frame from ASCII File

In [59]:
gaze = preprocessing.load_trial_level_raw_data(sid, trial_columns=settings.TRIAL_COLS, load_metadata=True)

ColumnNotFoundError: column position_x from position_columns is not available in samples dataframe

In [13]:
# inspect the gaze samples
gaze.samples.head()

time,pupil,page,trial,stimulus,pixel,position,velocity
i64,f64,str,str,str,list[f64],list[f64],list[f64]
4155117,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.0, 100.4]","[-15.702869, -10.555472]","[-0.22409, 0.76424]"
4155118,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 100.3]","[-15.695336, -10.558051]","[-0.16456, 0.792012]"
4155119,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 99.4]","[-15.695336, -10.581259]","[-0.044335, 0.746959]"
4155120,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.8, 100.9]","[-15.682779, -10.542577]","[-0.059119, 0.684403]"
4155121,540.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.7, 99.3]","[-15.68529, -10.583838]","[-0.118434, 0.704238]"


In [16]:
sess.pm_gaze_metadata = gaze._metadata

Eye movements are recorded in screen pixel coordinates, which depend on stimulus size and monitor setup. To compare gaze behavior across participants, screens, or datasets, it is standard to convert pixel positions 
into **degrees of visual angle (dva)**. Next, we compute **gaze velocity**, which allows us to detect saccades and distinguish them from fixations.

Now we save the gaze data and metadata we just extracted from the asc-file.

In [ ]:
sid.raw_data_dir, sid.metadata_dir

### Output directory structure

All preprocessed data is organised by data type under `preprocessed_data/<data_collection_name>/`. Each data type folder contains one subfolder per session:

```
preprocessed_data/<dcn>/
├── raw_data/
│   └── <session_save_name>/
├── fixations/
│   └── <session_save_name>/
├── saccades/
│   └── <session_save_name>/
├── scanpaths/
│   └── <session_save_name>/
├── reading_measures/
│   └── <session_save_name>/
├── sanity_checks/
│   └── <session_save_name>/
├── metadata/
│   └── <session_save_name>/
│       ├── gaze_metadata.json
│       ├── experiment.yaml
│       ├── calibrations.tsv
│       ├── calibrations.feather
│       ├── validations.tsv
│       ├── validations.feather
│       └── <session_idf>_overview.yaml
├── participant_data.csv
├── <dcn>_overview.yaml
└── stimuli_<dcn>/
```

The `Sid` object provides convenient properties to access each path:

In [ ]:
print(f"Raw data:        {sid.raw_data_dir}")
print(f"Metadata:        {sid.metadata_dir}")
print(f"Fixations:       {sid.fixations_dir}")
print(f"Saccades:        {sid.saccades_dir}")
print(f"Scanpaths:       {sid.scanpaths_dir}")
print(f"Reading measures: {sid.reading_measures_dir}")

In order to have the metadata which is extracted by pymovements available to create out session overview, we get this information from pymovements and store it in our session object.

In [17]:
sess.pm_gaze_metadata = gaze._metadata
sess.calibrations = gaze.calibrations
sess.validations = gaze.validations

## Stage 2a: Detect Events and Compute Their Properties

Eye-tracking data are typically segmented into events, i.e. `fixations` and `saccades`. Fixations represent moments when the eyes remain relatively still, allowing visual information to be processed, while saccades are the rapid movements between fixations that reposition the gaze. Detecting these events and computing their properties, such as `dispersion`, fixation `duration`, saccade `amplitude`, and `peak velocity`, provides the foundation for analyzing visual behavior and understanding how participants explore a stimulus.

### Fixations

We can detect fixations by applying the `I-VT` or the `I-DT` method.

The **I-VT (Velocity-Threshold Identification)** method distinguishes fixation and saccade points based on their point-to-point velocities. Each point is classified as a fixation if its velocity is below the specified threshold. Consecutive fixation points are then merged into a single fixation. A threshold of 20 degrees/second is commonly used as a default maximum value. Read more about [the IVT algorithm in the documentation](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.ivt.html) 

The **I-DT (Dispersion-Threshold Identification)** method finds fixations by grouping consecutive points within a maximum separation (dispersion) threshold and a minimum duration threshold. The algorithm slides a moving window across the data: if the dispersion within the window is below the threshold, the window represents a fixation and is gradually expanded until the dispersion exceeds the threshold.
Read more about [our implementation of the IDT method](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.idt.html).

We use the `I-VT` algorithm with the following key deafault parameters:
- `minimum duration`: 100 ms 
- `velocity threshold`: 20.0

Such properties as `location`, containing the centroid coordinates of each fixation, and `dispersion` will also be calculated.

In [18]:
gaze = preprocessing.load_trial_level_events_data(
                        gaze,
                        sess.sid,
                        event_type=settings.FIXATION,
                        file_pattern=None,
                    )

In [19]:
gaze = preprocessing.load_trial_level_events_data(
                        gaze,
                        sess.sid,
                        event_type=settings.SACCADE,
                        file_pattern=None,
                    )

### Saccades

Saccades are rapid eye movements that shift the point of fixation from one location to another. We detect saccades (or micro-saccades) from the velocity sequence of gaze data using the [microsaccades algorithm](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.microsaccades.html#pymovements.events.detection.microsaccades). This algorithm implements a noise-adaptive velocity threshold, meaning that the detection threshold automatically scales with the noise level of the velocity signal. Such properties as `amplitude` and `peak velocity` of the detected saccades will also be calcuated.

The key default parameters are:
- `threshold_factor`: Multiplier used to determine the velocity threshold relative to the noise level of the signal. The default value is 6. A higher factor makes the algorithm more conservative (detects fewer saccades), while a lower factor makes it more sensitive.
- `minimum_duration`: Defines how long a velocity peak must persist to be classified as a saccade. The duration is expressed in the same units as timesteps. If no timesteps are provided, the value refers to the number of samples (default = 6), which corresponds to about 12 ms at a 500 Hz sampling rate. Shorter events are ignored as noise. 

Save our events data.

In [20]:
str(sess.sid), sess.sid

('009_SV_CH_1_ET1',
 Sid(pid='009', lang='SV', country='CH', lab='1', session='ET1', session_id=1, postfix=''))

## Stage 2b: Map Fixations to AOIs

Once we have the fixations, we can map each of them to the AOIs of the stimulus. The resulting scanpath can then be saved. Note that this features is not yet completely finished.

In [21]:
stimulus = sess.stimuli[0]
stimulus

Stimulus(id=1, name='PopSci_MultiplEYE', type='experiment', pages=[StimulusPage(number=1, text='MultiplEYE-projektet\n\nNamnet ”MultiplEYE” är en ordlek som kombinerar ”multilingualism” eller ”multiple languages” med ”eye” från ”eye-tracking”. MultiplEYE är en COST-aktion finansierad av den Europeiska unionen. COST-aktioner är forskningsnätverk som stöds av European Cooperation in Science and Technology, förkortat COST. Som finansiär stöder COST vårt växande nätverk av forskare i och utanför Europa genom att ge ekonomiskt stöd till genomförandet av olika networkingevent.', image_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/stimuli_images_sv_ch_1/popsci_multipleye_id1_page_1_sv.png'), aoi_image_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/aoi_stimuli_images_sv_

In [22]:
stimulus.text_stimulus

char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,page,word_idx,word_idx_in_line,word
i64,str,f64,f64,i64,f64,i64,i64,str,i64,i64,str
0,"""M""",81.0,89.0,14,31.0,0,0,"""page_1""",0,0,"""MultiplEYE-projektet"""
1,"""u""",95.0,89.0,14,31.0,1,0,"""page_1""",0,0,"""MultiplEYE-projektet"""
2,"""l""",109.0,89.0,14,31.0,2,0,"""page_1""",0,0,"""MultiplEYE-projektet"""
3,"""t""",123.0,89.0,14,31.0,3,0,"""page_1""",0,0,"""MultiplEYE-projektet"""
4,"""i""",137.0,89.0,14,31.0,4,0,"""page_1""",0,0,"""MultiplEYE-projektet"""
…,…,…,…,…,…,…,…,…,…,…,…
649,"""t""",557.0,778.75,14,89.9,34,8,"""page_9""",89,5,"""text."""
650,"""e""",571.0,778.75,14,89.9,35,8,"""page_9""",89,5,"""text."""
651,"""x""",585.0,778.75,14,89.9,36,8,"""page_9""",89,5,"""text."""


In [ ]:
preprocessing.map_fixations_to_aois(gaze, sess.stimuli)

In [ ]:
# The resulting mapping can be stored as a scanpath, which is a sequence of AOIs that were fixated in the order they were fixated.
preprocessing.save_scanpaths(sid, gaze)

In [ ]:
# save metadata again
preprocessing.save_session_metadata(sid, gaze)

## Stage 3: Calculate AOI-based Measures

In this last step, we calculate the aoi-based measures. These are also refered to as reading measures, as they are typically used in reading research. They include measures such as first pass fixation duration (FPF), total fixation count (TFC), regression path duration (RPD), and many more. These measures are calculated based on the fixations that were mapped to the AOIs in the previous step.

### Fixation-based Metrics

As an intermediate step, the fixations are annotated. These annoataions include:
- The run ID. This ID specifies continuous sequences of fixations on the same word. It is used to calculate first pass and second pass measures.
- Whether the fixation is within the first pass or not
- The index of the preceding word and the following word
- If the saccade entering or leaving the fixation is a regression or not
- Whether it is the first fixation on the word or not

This information is necessary to calculate the reading measures in the next step.

In [ ]:
rm_df = preprocessing.calculate_reading_measures(gaze, sess.stimuli)
preprocessing.save_reading_measures(sid, rm_df)

In [ ]:
rm_df.filter(pl.col("trial") == "trial_1").head()

## Stage 4: Comprehension Question Answers
In addition to gaze data, each session contains answers to comprehension questions.
These are extracted from the ASC messages. The answers are matched to the stimulus order using the `question_order_versions.csv` file in the session's logfiles folder.

In [ ]:
answers_csv = sid.answers_dir / f"{sid}_answers.csv"
question_order_csv = (
    sess.session_folder_path / "logfiles" / "question_order_versions.csv"
)
parsed_answers = preprocessing.parse_answers_from_messages(gaze.messages)
source = "asc"

In [ ]:
parsed_answers

In [ ]:
preprocessing.collect_session_answers(
    question_order_csv=question_order_csv,
    stimuli_trial_map=sess.stimuli_trial_mapping,
    stimuli=sess.stimuli,
    parsed_answers=parsed_answers,
    out_path=answers_csv,
    source=source,
    completed_stimuli_ids=sess.completed_stimuli_ids,
)

## Final Steps

In the very end, we can create the session and dataset overview and store them as well. In addition, the participant data can be parsed and stored.

For the MultiplEYE data, there is also the option to create a sanity check report.

In [ ]:
multipleye.create_sanity_check_report(
    gaze,
    sess.session_identifier,
    output_dir=settings.OUTPUT_DIR,
    plotting=True,
    recalculate=True,
)

In [ ]:
multipleye.create_session_overview(sess.session_identifier, path=settings.OUTPUT_DIR)
multipleye.create_dataset_overview(path=settings.OUTPUT_DIR)
multipleye.parse_participant_data(settings.OUTPUT_DIR / "participant_data.csv")

In [ ]:
from preprocessing.psychometric_tests.preprocess_psychometric_tests import (
    preprocess_all_sessions,
)

preprocess_all_sessions()